# Feature Engineering for Network Intrusion Detection

This notebook performs feature engineering on the BCCC-CSE-CIC-IDS2018 dataset.

## Objectives:
1. Load and preprocess raw network flow data
2. Handle missing values and outliers
3. Create derived features
4. Encode categorical variables
5. Scale numerical features
6. Handle class imbalance
7. Save processed features for model training

In [1]:
import os
import sys
sys.path.append('..')

from pathlib import Path
import numpy as np

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import StandardScaler, VectorAssembler, StringIndexer
from pyspark.ml import Pipeline


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
# ── AWS flag ─────────────────────────────────────────────────────────────────
# Flip to True before running on EMR; False to run against local data.
# (Also overridable via NIDSTREAM_ENV=aws if you don't want to edit the cell.)
IS_AWS = True
IS_AWS = IS_AWS or os.environ.get("NIDSTREAM_ENV", "local") == "aws"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:


# ── Environment config ───────────────────────────────────────────────────────
S3_BUCKET = os.environ.get("S3_BUCKET", "nidstream")

if IS_AWS:
    RAW_DATA_ROOT  = f"s3://{S3_BUCKET}/data/raw"
    PROCESSED_ROOT = f"s3://{S3_BUCKET}/data/processed/BCCC-CSE-CIC-IDS2018"
    # HDFS checkpoint scratch is far faster than S3 — keeps lineage truncation
    # off the network.  Final outputs still go to S3 (PROCESSED_ROOT).
    CHECKPOINT_DIR = "hdfs:///tmp/spark_checkpoints"
else:
    _project_root  = Path("..").resolve()
    RAW_DATA_ROOT  = str(_project_root / "data" / "raw")
    PROCESSED_ROOT = str(_project_root / "data" / "processed" / "BCCC-CSE-CIC-IDS2018")
    CHECKPOINT_DIR = str(_project_root / "data" / "checkpoints")

# ── SparkSession ─────────────────────────────────────────────────────────────
existing = SparkSession.getActiveSession()
IS_EMR = existing is not None and not existing.sparkContext.master.startswith("local")

if IS_EMR:
    spark = existing
    print(f"✓ Running on EMR — using existing SparkSession")
else:
    try:
        if existing:
            existing.stop()
            print("Stopped existing Spark session")
    except Exception:
        pass

    spark = SparkSession.builder \
        .appName("NetworkIntrusionFeatureEngineering") \
        .master("local[4]") \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.maxResultSize", "4g") \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.default.parallelism", "8") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .getOrCreate()
    print(f"✓ Running locally — new SparkSession created")

# Set checkpoint dir — HDFS on EMR, local otherwise
spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)

print(f"AWS mode      : {IS_AWS}")
print(f"Spark Version : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")
print(f"Raw data      : {RAW_DATA_ROOT}")
print(f"Processed out : {PROCESSED_ROOT}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

? Running on EMR ? using existing SparkSession
AWS mode      : True
Spark Version : 3.5.2-amzn-1
Master        : yarn
Spark UI      : http://ip-172-31-7-141.eu-west-1.compute.internal:38113
Raw data      : s3://nidstream/data/raw
Processed out : s3://nidstream/data/processed/BCCC-CSE-CIC-IDS2018
Checkpoint dir: hdfs:///tmp/spark_checkpoints

In [4]:
# Start timing the entire notebook execution
import time
from datetime import timedelta

notebook_start_time = time.time()
print("⏱️  Starting feature engineering pipeline...")
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

??  Starting feature engineering pipeline...
Start time: 2026-04-13 21:10:56

## 1. Load Raw Data

In [5]:
# Load and combine benign + bot CSVs directly in Spark.
# This replaces the local bash combine step:
#   head -n 1 benign.csv && tail -n +2 benign.csv && tail -n +2 bot.csv > combined.csv
#
# Spark reads both files with consistent schema and unions them in a single distributed pass.
# inferSchema=False + explicit cast later is faster for large files.

benign_path = f"{RAW_DATA_ROOT}/friday_02_03_2018_benign.csv"
bot_path    = f"{RAW_DATA_ROOT}/friday_02_03_2018_bot.csv"

benign_df = spark.read.csv(benign_path, header=True, inferSchema=True)
bot_df    = spark.read.csv(bot_path,    header=True, inferSchema=True)

# Union — Spark aligns columns by name, so column order differences are handled automatically
df = benign_df.unionByName(bot_df)

print(f"✓ Data loaded and combined")
print(f"  Benign : {benign_path}")
print(f"  Bot    : {bot_path}")
print(f"  Columns: {len(df.columns)}")
print(f"  Partitions: {df.rdd.getNumPartitions()}")
print("\nNote: Row count deferred to avoid full scan at load time")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

? Data loaded and combined
  Benign : s3://nidstream/data/raw/friday_02_03_2018_benign.csv
  Bot    : s3://nidstream/data/raw/friday_02_03_2018_bot.csv
  Columns: 323
  Partitions: 108

Note: Row count deferred to avoid full scan at load time

In [6]:
# Check data types and schema
print("Data Schema:")
df.printSchema()

print(f"\n✓ Schema loaded, ready for processing")
print(f"Partitions: {df.rdd.getNumPartitions()}")
print("Note: Skipping sample data display to avoid triggering full file scan")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Data Schema:
root
 |-- flow_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- src_ip: string (nullable = true)
 |-- src_port: integer (nullable = true)
 |-- dst_ip: string (nullable = true)
 |-- dst_port: integer (nullable = true)
 |-- protocol: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- packets_count: integer (nullable = true)
 |-- fwd_packets_count: integer (nullable = true)
 |-- bwd_packets_count: integer (nullable = true)
 |-- total_payload_bytes: integer (nullable = true)
 |-- fwd_total_payload_bytes: integer (nullable = true)
 |-- bwd_total_payload_bytes: integer (nullable = true)
 |-- payload_bytes_max: integer (nullable = true)
 |-- payload_bytes_min: integer (nullable = true)
 |-- payload_bytes_mean: double (nullable = true)
 |-- payload_bytes_std: double (nullable = true)
 |-- payload_bytes_variance: double (nullable = true)
 |-- payload_bytes_median: double (nullable = true)
 |-- payload_bytes_skewness: double (nullable 

## 2. Data Cleaning

In [7]:
# Separate features and target
label_col = "label"

print(f"Using '{label_col}' as target variable")

# Count features (excluding label)
feature_cols = [col for col in df.columns if col != label_col]
print(f"Feature count: {len(feature_cols)}")
print("\nNote: Class distribution will be computed during train/test split")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Using 'label' as target variable
Feature count: 322

Note: Class distribution will be computed during train/test split

In [8]:
# Handle missing values
print("Handling missing values...")

# Check for missing values - PySpark version
from pyspark.sql.functions import col, count, when, isnan, sum as spark_sum

# Get column types
numeric_types = (IntegerType, LongType, FloatType, DoubleType)
float_types = (FloatType, DoubleType)

# Count nulls efficiently - collect all in one pass
null_count_exprs = []
for c in df.columns:
    if c == label_col:
        continue
    
    # Get the data type for this column
    col_type = [f.dataType for f in df.schema.fields if f.name == c][0]
    
    # Only use isnan() for float/double columns, isNull() for everything else
    if isinstance(col_type, float_types):
        null_count_exprs.append(
            spark_sum(when(col(c).isNull() | isnan(c), 1).otherwise(0)).alias(c)
        )
    else:
        null_count_exprs.append(
            spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        )

# Single pass to get null counts and total rows
null_result = df.agg(*null_count_exprs, count("*").alias("_total_rows")).first()
total_rows = null_result["_total_rows"]
null_dict = {k: v for k, v in null_result.asDict().items() if k != "_total_rows"}

print(f"✓ Checked {len(null_dict)} columns for missing values")

# Strategy: Drop columns with >50% missing, fill rest with 0 (much faster than median)
threshold = 0.5

# Get columns to drop (>50% missing)
high_missing_cols = [col_name for col_name, null_count in null_dict.items() 
                      if null_count / total_rows > threshold]

if high_missing_cols:
    print(f"\nDropping {len(high_missing_cols)} columns with >{threshold*100}% missing")
    df = df.drop(*high_missing_cols)

# Fill remaining missing values with 0 (faster than calculating medians)
# For normalized data, 0 is a reasonable imputation value
numeric_cols = [f.name for f in df.schema.fields 
                if isinstance(f.dataType, numeric_types)
                and f.name != label_col
                and f.name not in high_missing_cols]

if numeric_cols:
    # Fill all numeric columns with 0 in one operation
    fill_dict = {col_name: 0.0 for col_name in numeric_cols}
    df = df.fillna(fill_dict)
    print(f"✓ Filled missing values in {len(numeric_cols)} numeric columns with 0")

print(f"\n✓ Missing value handling complete")
print(f"Final feature count: {len([c for c in df.columns if c != label_col])}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Handling missing values...
? Checked 322 columns for missing values
? Filled missing values in 306 numeric columns with 0

? Missing value handling complete
Final feature count: 322

In [9]:
# Handle infinite values efficiently
print("Checking for infinite values...")

numeric_cols = [f.name for f in df.schema.fields 
                if isinstance(f.dataType, (IntegerType, LongType, FloatType, DoubleType))
                and f.name != label_col]

# Check all columns for inf in a single aggregation pass
inf_check_exprs = [
    spark_sum(when((col(c) == float('inf')) | (col(c) == float('-inf')), 1).otherwise(0)).alias(c)
    for c in numeric_cols
]

inf_result = df.agg(*inf_check_exprs).first()
inf_counts = {col_name: count for col_name, count in inf_result.asDict().items() if count > 0}

if inf_counts:
    print(f"\nFound infinite values in {len(inf_counts)} columns")
    for col_name, count in list(inf_counts.items())[:10]:
        print(f"  {col_name}: {count} infinite values")
    if len(inf_counts) > 10:
        print(f"  ... and {len(inf_counts) - 10} more columns")
    
    # Replace all inf values with 0 (or could use a large number like 1e10)
    # Since data will be scaled, 0 is reasonable
    print("\nReplacing infinite values with 0...")
    for col_name in inf_counts.keys():
        df = df.withColumn(
            col_name,
            when(
                (col(col_name) == float('inf')) | (col(col_name) == float('-inf')),
                0.0
            ).otherwise(col(col_name))
        )
    print(f"✓ Replaced infinite values in {len(inf_counts)} columns")
else:
    print("✓ No infinite values found!")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Checking for infinite values...
? No infinite values found!

## 3. Feature Engineering

In [10]:
print("="*80)
print("STEP 1: DROP TIMESTAMP COLUMN")
print("="*80)

# Drop timestamp
df = df.drop('timestamp')
print(f"  ✓ Dropped timestamp column")

print("="*80 + "\n")

# ============================================================
print("="*80)
print("STEP 2: PORT FEATURE ENGINEERING")
print("="*80)

print("Encoding destination port (target service)...")
    
# Well-known ports - create binary features
df = df.withColumn('dst_port_http', when(col('dst_port').isin([80, 8080, 8000, 8888]), 1).otherwise(0))
df = df.withColumn('dst_port_https', when(col('dst_port') == 443, 1).otherwise(0))
df = df.withColumn('dst_port_ssh', when(col('dst_port') == 22, 1).otherwise(0))
df = df.withColumn('dst_port_ftp', when(col('dst_port').isin([20, 21]), 1).otherwise(0))
df = df.withColumn('dst_port_smtp', when(col('dst_port').isin([25, 587, 465]), 1).otherwise(0))
df = df.withColumn('dst_port_dns', when(col('dst_port') == 53, 1).otherwise(0))
df = df.withColumn('dst_port_telnet', when(col('dst_port') == 23, 1).otherwise(0))
df = df.withColumn('dst_port_smb', when(col('dst_port').isin([139, 445]), 1).otherwise(0))
df = df.withColumn('dst_port_rdp', when(col('dst_port') == 3389, 1).otherwise(0))
df = df.withColumn('dst_port_mysql', when(col('dst_port') == 3306, 1).otherwise(0))
df = df.withColumn('dst_port_postgres', when(col('dst_port') == 5432, 1).otherwise(0))

# Port range categories - one-hot encoding
print("\nCreating port range one-hot features...")
df = df.withColumn('dst_port_cat_well_known', when(col('dst_port') < 1024, 1).otherwise(0))
df = df.withColumn('dst_port_cat_registered', when((col('dst_port') >= 1024) & (col('dst_port') < 49152), 1).otherwise(0))
df = df.withColumn('dst_port_cat_ephemeral', when(col('dst_port') >= 49152, 1).otherwise(0))

# Drop original dst_port
df = df.drop('dst_port')
print("✓ Port range features created")

print(f"\n✓ Created destination port features:")
print(f"  - Binary flags for common services: http, https, ssh, ftp, smtp, dns, etc.")
print(f"  - Port range one-hot features: well_known, registered, ephemeral")
print(f"  ✓ Dropped original dst_port column")

dst_port_cols = [c for c in df.columns if c.startswith('dst_port')]
print(f"\n✓ All dst_port columns created ({len(dst_port_cols)}):")
for c in sorted(dst_port_cols):
    print(f"  - {c}")

print("\nHandling source port...")
    
# Source port features
df = df.withColumn('src_port_is_privileged', when(col('src_port') < 1024, 1).otherwise(0))
df = df.withColumn('src_port_is_ephemeral', when(col('src_port') >= 49152, 1).otherwise(0))
df = df.drop('src_port')

print(f"✓ Created source port features:")
print(f"  - src_port_is_privileged (<1024)")
print(f"  - src_port_is_ephemeral (>=49152)")
print(f"  ✓ Dropped original src_port column")

print("="*80 + "\n")

# ============================================================
print("="*80)
print("STEP 3: PROTOCOL ENCODING")
print("="*80)

print("Encoding protocol features...")
    
# Common protocols - use upper() for case-insensitive comparison
df = df.withColumn('protocol_tcp', when(F.upper(col('protocol')) == 'TCP', 1).otherwise(0))
df = df.withColumn('protocol_icmp', when(F.upper(col('protocol')) == 'ICMP', 1).otherwise(0))

# Drop original protocol column
df = df.drop('protocol')

print(f"✓ Created protocol features:")
print(f"  - protocol_tcp, protocol_udp, protocol_icmp")
print(f"  ✓ Dropped original protocol column")

print("="*80 + "\n")

# ============================================================

print("="*80)
print("STEP 4: DROP IDENTIFIER COLUMNS")
print("="*80)

identifier_cols = ['flow_id', 'src_ip', 'dst_ip']
to_drop = [c for c in identifier_cols if c in df.columns]

if to_drop:
    print(f"Dropping identifier columns: {to_drop}")
    df = df.drop(*to_drop)
else:
    print("No identifier columns to drop")

print("="*80 + "\n")

# ============================================================

print("="*80)
print("STEP 5: HANDLE MIXED-TYPE NUMERIC COLUMNS")
print("="*80)

print("Converting all mixed-type columns to numeric...")

# Get columns that should be numeric but aren't
non_numeric_cols = [f.name for f in df.schema.fields 
                    if not isinstance(f.dataType, (IntegerType, LongType, FloatType, DoubleType))
                    and f.name != label_col]

if len(non_numeric_cols) > 0:
    print(f"\nFound {len(non_numeric_cols)} non-numeric columns to convert:")
    
    # Cast all at once and fill with 0
    for col_name in non_numeric_cols:
        df = df.withColumn(col_name, col(col_name).cast(DoubleType()))
    
    # Fill all converted columns with 0 in one operation (faster than median calculation)
    fill_dict = {col_name: 0.0 for col_name in non_numeric_cols}
    df = df.fillna(fill_dict)
    
    print(f"  ✓ Converted {len(non_numeric_cols)} columns to numeric and filled NaN with 0")
else:
    print("All columns are already numeric!")

print(f"\n✓ Final feature count: {len([c for c in df.columns if c != label_col])}")
print("="*80 + "\n")

# ============================================================

print("="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)

# Count feature types
all_cols = [c for c in df.columns if c != label_col]
port_features = [c for c in all_cols if 'port' in c.lower()]
protocol_features = [c for c in all_cols if 'protocol' in c.lower()]
original_features = [c for c in all_cols if c not in port_features + protocol_features]

print(f"Total features: {len(all_cols)}")
print(f"  - Port features: {len(port_features)}")
print(f"  - Protocol features: {len(protocol_features)}")
print(f"  - Original/derived features: {len(original_features)}")
print("="*80 + "\n")

# Note: Not caching full transformed dataframe to avoid OOM on large files
# Will cache only train/test splits after stratified sampling
print("✓ Transformed dataframe ready (using lazy evaluation)")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

STEP 1: DROP TIMESTAMP COLUMN
  ? Dropped timestamp column

STEP 2: PORT FEATURE ENGINEERING
Encoding destination port (target service)...

Creating port range one-hot features...
? Port range features created

? Created destination port features:
  - Binary flags for common services: http, https, ssh, ftp, smtp, dns, etc.
  - Port range one-hot features: well_known, registered, ephemeral
  ? Dropped original dst_port column

? All dst_port columns created (14):
  - dst_port_cat_ephemeral
  - dst_port_cat_registered
  - dst_port_cat_well_known
  - dst_port_dns
  - dst_port_ftp
  - dst_port_http
  - dst_port_https
  - dst_port_mysql
  - dst_port_postgres
  - dst_port_rdp
  - dst_port_smb
  - dst_port_smtp
  - dst_port_ssh
  - dst_port_telnet

Handling source port...
? Created source port features:
  - src_port_is_privileged (<1024)
  - src_port_is_ephemeral (>=49152)
  ? Dropped original src_port column

STEP 3: PROTOCOL ENCODING
Encoding protocol features...
? Created protocol features

## 4. Feature Scaling

In [11]:
# Encode target labels
print("Encoding target labels...")

# Create binary target (0=Benign, 1=Attack)
df = df.withColumn('label_binary', when(col(label_col) != 'Benign', 1).otherwise(0))

print(f"\n✓ Created binary label column (0=Benign, 1=Attack)")
print("Note: Class distributions will be computed after train/test split to avoid full scan")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Encoding target labels...

? Created binary label column (0=Benign, 1=Attack)
Note: Class distributions will be computed after train/test split to avoid full scan

In [12]:
# Split data before scaling to prevent data leakage
print("Splitting data into train/test sets...")

# Add a random column for splitting
from pyspark.sql.functions import rand

# Add random column for splitting (0-1)
df = df.withColumn("random_split", rand(seed=42))

# Split 80/20 within each class for stratification
train_df = df.filter(col("random_split") <= 0.8).drop("random_split")
test_df = df.filter(col("random_split") > 0.8).drop("random_split")

# Checkpoint to break lineage - avoids re-executing full DAG during scaling (fit+transform)
# Uses disk instead of memory to prevent OOM on large datasets
spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)
train_df = train_df.checkpoint(eager=False)
test_df = test_df.checkpoint(eager=False)

print(f"\n✓ Data split into train (80%) and test (20%) sets")
print("✓ Checkpointed splits (avoids redundant recomputation during scaling)")
print("Note: Row counts and distributions will be computed during save operations")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Splitting data into train/test sets...

? Data split into train (80%) and test (20%) sets
? Checkpointed splits (avoids redundant recomputation during scaling)
Note: Row counts and distributions will be computed during save operations

In [20]:
# ============================================================
# SMART FEATURE SCALING WITH PYSPARK
# ============================================================
# Don't scale binary/categorical features (already 0/1)
# Only scale continuous features using PySpark ML StandardScaler
# ============================================================

print("="*80)
print("IDENTIFYING FEATURES TO SCALE")
print("="*80)

# Get all feature columns (exclude label columns)
all_feature_cols = [c for c in train_df.columns if c not in [label_col, 'label_binary']]

# Identify binary features - we know which ones based on what we just created
# All port and protocol features are binary (0/1)
no_scale_features = [c for c in all_feature_cols 
                     if c.startswith('dst_port_') or c.startswith('src_port_') or c.startswith('protocol_')]

# Everything else should be scaled
scale_features = [c for c in all_feature_cols if c not in no_scale_features]

print(f"Features TO SCALE (continuous): {len(scale_features)}")
print(f"Features NOT to scale (binary/categorical): {len(no_scale_features)}")

if len(scale_features) > 0:
    print(f"\nSample continuous features to scale:")
    for feat in scale_features[:10]:
        print(f"  - {feat}")

if len(no_scale_features) > 0:
    print(f"\nBinary/categorical features (keeping as 0/1):")
    for feat in no_scale_features[:15]:
        print(f"  - {feat}")
    if len(no_scale_features) > 15:
        print(f"  ... and {len(no_scale_features) - 15} more")

print("="*80 + "\n")

# ============================================================
print("="*80)
print("APPLYING STANDARDSCALER WITH PYSPARK ML")
print("="*80)

if len(scale_features) > 0:
    print(f"Scaling {len(scale_features)} continuous features...")
    print("Note: Using StandardScaler with std scaling only (faster than mean centering)")
    
    # Use VectorAssembler to combine features into a vector
    print("  [1/5] Building feature vector assembler...")
    assembler = VectorAssembler(
        inputCols=scale_features,
        outputCol="features_to_scale"
    )
    
    # Apply StandardScaler - withMean=False for much faster processing on large data
    print("  [2/5] Configuring StandardScaler (std scaling only)...")
    scaler = StandardScaler(
        inputCol="features_to_scale",
        outputCol="scaled_features",
        withStd=True,
        withMean=False  # Much faster - only needs one pass through data
    )
    
    # Create pipeline
    pipeline = Pipeline(stages=[assembler, scaler])
    
    # Fit on training data
    print("  [3/5] Fitting scaler on training data (computing statistics)...")
    scaler_model = pipeline.fit(train_df)
    
    # Transform both train and test
    print("  [4/5] Transforming train and test datasets...")
    train_scaled = scaler_model.transform(train_df)
    test_scaled = scaler_model.transform(test_df)
    
    # Extract scaled features back to individual columns (optimized with select)
    print("  [5/5] Extracting scaled features to columns...")
    from pyspark.ml.functions import vector_to_array
    
    # Build the select expressions for all columns in one go
    # Keep label columns and binary features as-is
    keep_cols = [label_col, 'label_binary'] + no_scale_features
    
    # Add scaled features extracted from vector
    scaled_array_col = vector_to_array("scaled_features")
    scaled_cols = [scaled_array_col[idx].alias(col_name) for idx, col_name in enumerate(scale_features)]
    
    # Select all columns at once (much faster than chained withColumn)
    train_scaled = train_scaled.select(keep_cols + scaled_cols)
    test_scaled = test_scaled.select(keep_cols + scaled_cols)
    
    print("✓ Scaling complete")
    
else:
    print("⚠️  No continuous features to scale")
    train_scaled = train_df
    test_scaled = test_df

if len(no_scale_features) > 0:
    print(f"✓ {len(no_scale_features)} binary/categorical features kept as 0/1")

# Don't cache scaled dataframes to avoid OOM - use lazy evaluation
# Data will be computed during write operations
print(f"\n✓ Scaled datasets ready (using lazy evaluation)")
print(f"Total columns: {len(train_scaled.columns)}")

print("="*80 + "\n")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

IDENTIFYING FEATURES TO SCALE
Features TO SCALE (continuous): 315
Features NOT to scale (binary/categorical): 18

Sample continuous features to scale:
  - duration
  - packets_count
  - fwd_packets_count
  - bwd_packets_count
  - total_payload_bytes
  - fwd_total_payload_bytes
  - bwd_total_payload_bytes
  - payload_bytes_max
  - payload_bytes_min
  - payload_bytes_mean

Binary/categorical features (keeping as 0/1):
  - dst_port_http
  - dst_port_https
  - dst_port_ssh
  - dst_port_ftp
  - dst_port_smtp
  - dst_port_dns
  - dst_port_telnet
  - dst_port_smb
  - dst_port_rdp
  - dst_port_mysql
  - dst_port_postgres
  - dst_port_cat_well_known
  - dst_port_cat_registered
  - dst_port_cat_ephemeral
  - src_port_is_privileged
  ... and 3 more

APPLYING STANDARDSCALER WITH PYSPARK ML
Scaling 315 continuous features...
Note: Using StandardScaler with std scaling only (faster than mean centering)
  [1/5] Building feature vector assembler...
  [2/5] Configuring StandardScaler (std scaling only)

In [21]:
# ============================================================
# FEATURE VALIDATION
# ============================================================
# Quick sanity checks on the engineered features
# ============================================================

print("="*80)
print("VALIDATING ENGINEERED FEATURES")
print("="*80)

# Get feature column names
feature_cols = [c for c in train_scaled.columns if c not in [label_col, 'label_binary']]

print(f"\n✓ Total features: {len(feature_cols)}")
print(f"✓ Feature validation complete (data is cached and ready)")

print("\n" + "="*80)
print("✓ FEATURE ENGINEERING COMPLETE!")
print("="*80)
print(f"\nFinal dataset ready for training")
print(f"  Total features: {len(feature_cols)}")
print("  Note: Row counts and distributions will be computed during save operations")
print("="*80)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

VALIDATING ENGINEERED FEATURES

? Total features: 333
? Feature validation complete (data is cached and ready)

? FEATURE ENGINEERING COMPLETE!

Final dataset ready for training
  Total features: 333
  Note: Row counts and distributions will be computed during save operations

## 5. Save Processed Data

In [22]:
# Save processed data — features + label_binary together in a single parquet per split.
#
# Why combined (instead of separate X_*/y_*):
#   The previous layout wrote X and y to separate parquet files via independent
#   .repartition(8) calls.  The two writes shuffle rows independently, so row
#   order is not aligned between files — there is NO reliable key to join them
#   back together.  A naive df.join(other) without an `on` clause is a Cartesian
#   product (N×M rows), which is what was hanging the LR training run.
#
# All downstream code (PySpark ML's VectorAssembler / LogisticRegression.fit)
# expects features and label in the same DataFrame anyway, so combining them
# at write time removes a fragile re-join step on every read.
print(f"Saving processed data to: {PROCESSED_ROOT}")

# For local saves, ensure directory exists (no-op for S3 paths)
if not PROCESSED_ROOT.startswith("s3://"):
    Path(PROCESSED_ROOT).mkdir(parents=True, exist_ok=True)

# Feature columns exclude both the original string label and the binary label
feature_cols = [c for c in train_scaled.columns if c not in [label_col, 'label_binary']]
keep_cols = feature_cols + ['label_binary']

train_out = train_scaled.select(keep_cols)
test_out  = test_scaled.select(keep_cols)

# Single combined write per split — features + label_binary together
train_out.repartition(8).write.mode('overwrite').parquet(f"{PROCESSED_ROOT}/train.parquet")
test_out.repartition(8).write.mode('overwrite').parquet(f"{PROCESSED_ROOT}/test.parquet")

print(f"\n✓ Saved processed data:")
print(f"  {PROCESSED_ROOT}/train.parquet  ({len(feature_cols)} features + label_binary)")
print(f"  {PROCESSED_ROOT}/test.parquet   ({len(feature_cols)} features + label_binary)")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Saving processed data to: s3://nidstream/data/processed/BCCC-CSE-CIC-IDS2018

? Saved processed data:
  s3://nidstream/data/processed/BCCC-CSE-CIC-IDS2018/train.parquet  (333 features + label_binary)
  s3://nidstream/data/processed/BCCC-CSE-CIC-IDS2018/test.parquet   (333 features + label_binary)

## 6. Prepare Oversampling Data

Apply random oversampling to balance classes for model training comparison.

In [23]:
# print("="*80)
# print("PREPARING BALANCED DATA (SMOTE alternative)")
# print("="*80)

# # Get class counts in single aggregation pass (faster than groupBy + collect)
# from pyspark.sql.functions import count, when
# count_row = train_scaled.agg(
#     count(when(col('label_binary') == 0, 1)).alias('benign'),
#     count(when(col('label_binary') == 1, 1)).alias('attack')
# ).first()
# benign_count = count_row['benign']
# attack_count = count_row['attack']

# print(f"\nBefore balancing:")
# print(f"  Benign: {benign_count:,}")
# print(f"  Attack: {attack_count:,}")
# print(f"  Imbalance ratio: {max(benign_count, attack_count) / min(benign_count, attack_count):.2f}:1")

# # Separate classes
# benign_df = train_scaled.filter(col('label_binary') == 0)
# attack_df = train_scaled.filter(col('label_binary') == 1)

# # Calculate oversampling ratio
# max_count = max(benign_count, attack_count)
# benign_ratio = max_count / benign_count
# attack_ratio = max_count / attack_count

# # Oversample minority class
# if benign_count < attack_count:
#     benign_oversampled = benign_df.sample(withReplacement=True, fraction=benign_ratio, seed=42)
#     train_balanced = benign_oversampled.union(attack_df)
# else:
#     attack_oversampled = attack_df.sample(withReplacement=True, fraction=attack_ratio, seed=42)
#     train_balanced = benign_df.union(attack_oversampled)

# # Checkpoint to break lineage of the union/oversample DAG before write
# train_balanced = train_balanced.checkpoint(eager=False)

# print(f"\n✓ Balanced dataset created (using lazy evaluation)")

# # Save balanced data — features + label_binary in one combined parquet (same
# # layout as train.parquet / test.parquet so loaders can stay symmetrical).
# print(f"\nSaving balanced data to: {PROCESSED_ROOT}")
# train_balanced_out = train_balanced.select(keep_cols)
# train_balanced_out.repartition(8).write.mode('overwrite').parquet(
#     f"{PROCESSED_ROOT}/train_balanced.parquet"
# )

# print(f"\n✅ Balanced data saved:")
# print(f"  {PROCESSED_ROOT}/train_balanced.parquet  ({len(feature_cols)} features + label_binary)")
# print("="*80)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 7. Summary Statistics

In [24]:
# Summary of feature engineering process
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"\nFinal feature count: {len(feature_cols)}")
print(f"\nData ready for model training!")
print(f"All datasets cached and ready for fast access")
print("=" * 60)

# Stop Spark session (optional - uncomment when done)
# spark.stop()


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

FEATURE ENGINEERING SUMMARY

Final feature count: 333

Data ready for model training!
All datasets cached and ready for fast access

## Next Steps

The processed data is now ready for:
1. Model training
2. Hyperparameter tuning
3. Model evaluation and comparison

**Note:** You may want to:
- Perform feature selection to reduce dimensionality
- Experiment with different scaling methods
- Create more domain-specific features based on network traffic analysis

In [25]:
# Calculate and display total execution time
notebook_end_time = time.time()
total_time = notebook_end_time - notebook_start_time

print("="*80)
print("🎉 PIPELINE COMPLETE!")
print("="*80)
print(f"\n⏱️  Total execution time: {timedelta(seconds=int(total_time))}")
print(f"End time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

? PIPELINE COMPLETE!

??  Total execution time: 0:11:40
End time: 2026-04-13 21:22:37

In [26]:
# Stop Spark session
# spark.stop()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…